# 17 · Principal Component Analysis — Predictive Maintenance for Industrial Motors

> *"Ten sensors produce noise.  
> Three principal components produce understanding.  
> The distance in that reduced space tells you what the machine cannot say out loud."*

---

## 🏭 Business Context

An industrial motor in continuous production doesn't fail in isolation.  
It announces its decline across multiple channels simultaneously:
vibration climbs, bearing temperature rises, lubrication flow drops,  
current increases, power factor degrades.

A maintenance engineer watching 10 sensors on 10 separate trend charts  
will not catch the pattern until it's too late.  
The signal is distributed. The insight is in the correlation.

**Principal Component Analysis (PCA)** compresses 10 sensor readings into  
2–3 orthogonal axes that explain the dominant variation patterns in the data.  
The result is a **health space**: a low-dimensional map where:

- **Normal motors** cluster near the origin
- **Degrading motors** drift along PC1 (the mechanical health axis)
- **Failing motors** appear as outliers in SPE or Hotelling T² space

This notebook applies PCA to **1,500 motor condition snapshots** from 10 sensors,  
derives a **Reliability Index (IR)** that combines reconstruction error and Mahalanobis distance,  
and deploys a real-time **motor health classifier** — all without any labeled failure data.

---

**Project:** 17 | **Algorithm:** PCA | **Domain:** Predictive Maintenance / Reliability Engineering  
**Family:** Unsupervised Learning · Dimensionality Reduction  
**Status:** 📦 Paid Project — [lozanolsa.gumroad.com](https://lozanolsa.gumroad.com)

---

## ⚙️ Section 2 — Setup

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── Display ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

# ── LozanoLsa palette ────────────────────────────────────────────────────────
C_BLUE   = "#4C9BE8"   # primary brand
C_RED    = "#E8574C"   # critical / alert
C_GREEN  = "#3FB950"   # normal / safe
C_AMBER  = "#F0A84D"   # caution
C_ORANGE = "#F08C3A"   # severe
C_GRAY   = "#8b949e"   # muted

STATUS_COLORS = {"Normal": C_GREEN, "Alert": C_AMBER, "Critical": C_RED}
IR_COLORS     = {"Normal": C_GREEN, "Alert": C_AMBER, "Severe": C_ORANGE, "Critical": C_RED}

# IR thresholds (calibrated to dataset distribution)
IR_THRESHOLDS = {"Normal": 0.86, "Alert": 0.74, "Severe": 0.62}

# Aesthetic defaults
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

print("✅ Environment ready — Project 17 · PCA Predictive Maintenance")

## 📂 Section 3 — Load Data

**Dataset:** `mtto_data.csv` — 1,500 motor condition snapshots from 10 continuous sensors.

| Column | Unit | Description | PCA role |
|---|---|---|---|
| `vibration_rms_mm_s` | mm/s | RMS vibration amplitude | Mechanical health |
| `vibration_hf_g` | g | High-frequency vibration (bearing wear) | Mechanical health |
| `bearing_temp_c` | °C | Bearing housing temperature | Thermal health |
| `motor_current_a` | A | Motor phase current | Electrical load |
| `voltage_v` | V | Supply voltage | Electrical supply |
| `power_factor_pct` | % | Power factor | Electrical quality |
| `speed_rpm` | rpm | Shaft rotational speed | Mechanical load |
| `lube_flow_l_min` | L/min | Lubricant flow rate | Lubrication system |
| `lube_pressure_bar` | bar | Lubricant line pressure | Lubrication system |
| `internal_humidity_pct` | % | Internal motor humidity | Environmental |
| `op_status` | label | Operating state (Normal/Alert/Critical) | **Visualization only — not used in PCA** |

> ⚠️ `op_status` is used **only for EDA coloring and external validation**.  
> PCA operates in fully unsupervised mode on the 10 sensor readings.

In [ ]:
# ── Portable loader ──────────────────────────────────────────────────────────
try:
    df = pd.read_csv("mtto_data.csv")
except FileNotFoundError:
    df = pd.read_csv(
        "https://raw.githubusercontent.com/LozanoLsa/"
        "PCA_Predictive_Maintenance/main/mtto_data.csv"
    )

FEATURES = [
    "vibration_rms_mm_s", "vibration_hf_g", "bearing_temp_c",
    "motor_current_a", "voltage_v", "power_factor_pct", "speed_rpm",
    "lube_flow_l_min", "lube_pressure_bar", "internal_humidity_pct"
]

print(f"Dataset shape    : {df.shape}")
print(f"Sensor features  : {len(FEATURES)}")
print(f"Motor snapshots  : {len(df):,}")
print(f"Missing values   : {df.isnull().sum().sum()}")
print(f"\nOperating status distribution:")
print(df["op_status"].value_counts().to_string())
df.head()

## 🔍 Section 4 — Sanity Checks

PCA is sensitive to scale (corrected by StandardScaler) and to outliers  
(a single extreme sensor reading can distort the principal components).  
We verify physical plausibility and flag any values beyond ±4σ.

In [ ]:
# ── Descriptive statistics ───────────────────────────────────────────────────
print("=" * 60)
print("  Descriptive Statistics — Motor Sensor Readings")
print("=" * 60)
desc = df[FEATURES].describe().T
desc["cv_%"] = (desc["std"] / desc["mean"] * 100).round(1)
display(desc.round(2))

# ── Physical bounds ────────────────────────────────────────────────────────────
bounds = {
    "vibration_rms_mm_s":   (0.1, 12.0),
    "vibration_hf_g":       (0.01, 4.0),
    "bearing_temp_c":       (30, 130),
    "motor_current_a":      (0.5, 45),
    "voltage_v":            (430, 490),
    "power_factor_pct":     (70, 100),
    "speed_rpm":            (500, 1600),
    "lube_flow_l_min":      (5, 50),
    "lube_pressure_bar":    (0.1, 5.5),
    "internal_humidity_pct":(5, 95),
}
print("\n── Physical range check ────────────────────────────────")
violations = 0
for col, (lo, hi) in bounds.items():
    out = df[(df[col] < lo) | (df[col] > hi)].shape[0]
    if out > 0:
        print(f"  ⚠️  {col}: {out} values outside [{lo}, {hi}]")
        violations += 1
if violations == 0:
    print("  ✅ All sensors within physically plausible ranges.")

# ── Extreme value scan ────────────────────────────────────────────────────────
print("\n── Extreme value scan (>4σ from mean) ──────────────────")
for col in FEATURES:
    mu, sigma = df[col].mean(), df[col].std()
    extreme = ((df[col] - mu).abs() > 4 * sigma).sum()
    flag = "⚠️ " if extreme > 5 else "  "
    print(f"  {flag}{col}: {extreme} values beyond ±4σ")

## 📊 Section 5 — Exploratory Data Analysis

Three visualizations motivate the PCA approach:

1. **Correlation heatmap** — the sensors are not independent; high mutual correlation  
   means PCA can compress them effectively without information loss.
2. **Feature distributions by operating status** — confirms that the degradation signal  
   is distributed across multiple sensors simultaneously.
3. **Bivariate scatter (vibration vs temperature)** — shows partial separation  
   that PCA will sharpen by combining all 10 dimensions.

In [ ]:
# ── EDA 1 · Correlation heatmap ──────────────────────────────────────────────
corr = df[FEATURES].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.4, linecolor="#e0e0e0",
    annot_kws={"size": 8},
    cbar_kws={"shrink": 0.8}, ax=ax
)
ax.set_title(
    "Sensor Correlation Matrix — 10 Motor Condition Variables",
    fontsize=13, fontweight="bold", pad=14
)
plt.tight_layout()
plt.show()

print("── Key correlations ─────────────────────────────────────")
pairs = [
    ("vibration_rms_mm_s", "bearing_temp_c"),
    ("vibration_rms_mm_s", "lube_pressure_bar"),
    ("vibration_rms_mm_s", "power_factor_pct"),
    ("bearing_temp_c",    "motor_current_a"),
    ("lube_flow_l_min",   "lube_pressure_bar"),
]
for a, b in pairs:
    r = corr.loc[a, b]
    print(f"  {a} ↔ {b}: {r:+.3f}")
print()
print("  → Strong multi-collinearity confirmed across vibration, temperature,")
print("    current, and lubrication variables.")
print("  → PCA will compress this redundancy into a small number of latent axes.")

In [ ]:
# ── EDA 2 · Feature distributions by operating status ─────────────────────────
key_features = [
    "vibration_rms_mm_s", "bearing_temp_c",
    "lube_flow_l_min", "power_factor_pct"
]
fig, axes = plt.subplots(1, 4, figsize=(15, 4.5))
order = ["Normal", "Alert", "Critical"]
pal = [C_GREEN, C_AMBER, C_RED]

for ax, feat in zip(axes, key_features):
    sns.boxplot(
        data=df, x="op_status", y=feat, order=order,
        palette=pal, width=0.5,
        flierprops={"marker": ".", "alpha": 0.3, "markersize": 4},
        ax=ax
    )
    ax.set_xlabel("", fontsize=0)
    ax.set_title(feat.replace("_", " ").title(), fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", labelsize=9, rotation=10)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Sensor Distributions by Operating Status\n"
    "(Status labels used for visualization only — not used in PCA)",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

print("── Degradation signatures ────────────────────────────────")
print("  Normal → Critical:")
for f in key_features:
    n_mean = df[df["op_status"]=="Normal"][f].mean()
    c_mean = df[df["op_status"]=="Critical"][f].mean()
    delta  = (c_mean - n_mean) / n_mean * 100
    sign   = "+" if delta > 0 else ""
    print(f"  {f:<28}: {n_mean:.1f} → {c_mean:.1f} ({sign}{delta:.1f}%)")

In [ ]:
# ── EDA 3 · Multivariate scatter — vibration vs temperature ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (x_feat, y_feat, xl, yl) in zip(axes, [
    ("vibration_rms_mm_s", "bearing_temp_c",    "Vibration RMS (mm/s)", "Bearing Temp (°C)"),
    ("lube_pressure_bar",  "vibration_rms_mm_s", "Lube Pressure (bar)",  "Vibration RMS (mm/s)"),
]):
    for status, color in STATUS_COLORS.items():
        sub = df[df["op_status"] == status]
        ax.scatter(sub[x_feat], sub[y_feat],
                   c=color, label=status, alpha=0.45, s=14, edgecolors="none")
    ax.set_xlabel(xl, fontsize=10)
    ax.set_ylabel(yl, fontsize=10)
    ax.legend(title="Status", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_title("Vibration vs Temperature — Partial Separation",
                  fontsize=12, fontweight="bold")
axes[1].set_title("Lubrication Pressure vs Vibration — Degradation Gradient",
                  fontsize=12, fontweight="bold")

plt.suptitle(
    "Bivariate Views — The Status Clusters Partially Overlap\n"
    "PCA will sharpen this separation by using all 10 dimensions simultaneously",
    fontsize=11, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

print("  → No 2D view cleanly separates the three states.")
print("  → The discriminating signal is distributed across all 10 sensors.")
print("  → PCA compresses all 10 dimensions into a 2–3D health space.")

## 🔧 Section 6 — Preprocessing

PCA maximizes variance — but variance is scale-dependent.  
Without scaling, `voltage_v` (range ~430–490) would dominate over  
`vibration_hf_g` (range ~0–3) simply because of its magnitude.

**`StandardScaler`** transforms each sensor to zero mean and unit variance,  
ensuring each variable contributes equally to the principal components.

> No train/test split. PCA is applied to the full dataset —  
> the goal is to learn the normal operating manifold from all historical data.

In [ ]:
# ── Feature matrix + scaling ─────────────────────────────────────────────────
X = df[FEATURES].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix   : {X.shape}")
print(f"Scaled matrix    : {X_scaled.shape}")
print()
print("Post-scaling verification (all should be ≈ 0 mean, ≈ 1 std):")
for i, feat in enumerate(FEATURES):
    print(f"  {feat:<30}: mean={X_scaled[:,i].mean():+.6f} | std={X_scaled[:,i].std():.6f}")

## 🔢 Section 7 — Dimensionality Reduction: Scree Plot & Cumulative Variance

We run PCA on all 10 components first to diagnose the variance structure.  
The **scree plot** shows how much variance each component captures.  
We choose the number of retained components by the **90% cumulative variance rule** —  
or the 'elbow' where marginal gain flattens.

**Key result preview:**
- PC1 alone captures **54.4%** of total variance — the dominant health axis
- 3 PCs capture **74.0%** — sufficient for health monitoring
- 6 PCs needed for **90%** — retained for reconstruction error (SPE)

In [ ]:
# ── Full PCA (all 10 components) ─────────────────────────────────────────────
pca_full = PCA(n_components=len(FEATURES))
X_pca_full = pca_full.fit_transform(X_scaled)

var_exp = pca_full.explained_variance_ratio_
var_cum = np.cumsum(var_exp)

# ── Scree plot + cumulative variance ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
components = list(range(1, len(FEATURES) + 1))

# Left: Scree
ax1.bar(components, var_exp * 100, color=C_BLUE, alpha=0.8, edgecolor="white")
ax1.plot(components, var_exp * 100, "o-", color=C_RED, lw=2, ms=8)
ax1.axvline(x=3, color=C_AMBER, ls="--", lw=1.5, label="Chosen k = 3")
for i, v in enumerate(var_exp * 100):
    ax1.text(i + 1, v + 0.5, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
ax1.set_xlabel("Principal Component", fontsize=11)
ax1.set_ylabel("Variance Explained (%)", fontsize=11)
ax1.set_title("Scree Plot — Individual Variance per PC",
              fontsize=12, fontweight="bold")
ax1.legend(fontsize=9)
ax1.set_xticks(components)
ax1.grid(axis="y", alpha=0.3)

# Right: Cumulative
ax2.plot(components, var_cum * 100, "o-", color=C_BLUE, lw=2.5, ms=8)
for thresh, color, label in [(0.75, C_AMBER, "75%"), (0.90, C_GREEN, "90%"), (0.95, C_RED, "95%")]:
    ax2.axhline(y=thresh * 100, color=color, ls="--", lw=1.3, alpha=0.8, label=f"{label} threshold")
ax2.axvline(x=3, color=C_AMBER, ls="--", lw=1.5)
for i, v in enumerate(var_cum * 100):
    ax2.text(i + 1, v + 1, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
ax2.set_xlabel("Number of Components", fontsize=11)
ax2.set_ylabel("Cumulative Variance Explained (%)", fontsize=11)
ax2.set_title("Cumulative Variance — How many PCs to keep?",
              fontsize=12, fontweight="bold")
ax2.legend(fontsize=9, loc="lower right")
ax2.set_ylim(50, 105)
ax2.set_xticks(components)
ax2.grid(True, alpha=0.3)

plt.suptitle("PCA Variance Decomposition — 10-Sensor Motor Dataset",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("── Component selection summary ──────────────────────────")
for n in [3, 6, 7]:
    print(f"  {n} PCs → {var_cum[n-1]*100:.1f}% variance explained")
print()
print("  → We retain k=3 for the health space visualization and IR computation.")
print("  → PC1 captures 55.3% alone — it IS the motor health axis.")

## 🤖 Section 8 — Model Training: PCA Health Space

With **k = 3 components** selected, we fit the final PCA model and:
1. Project all 1,500 observations into the 3D health space
2. Compute the **Reliability Index (IR)** for each motor snapshot
3. Visualize the PC1–PC2 health map colored by operating status

**Physical interpretation of principal components:**

| Component | Variance | Interpretation | Key Loadings |
|---|---|---|---|
| **PC1** | 54.4% | Mechanical Degradation Axis | vibration (+), temp (+), current (+), lube (−) |
| **PC2** | 10.0% | Electrical Supply Axis | voltage (+0.97) |
| **PC3** | 9.6% | Speed Axis | speed_rpm (+0.92) |

In [ ]:
# ── Final PCA model (k=3 for health space) ────────────────────────────────────
K_PCA = 3
pca = PCA(n_components=K_PCA)
X_pca = pca.fit_transform(X_scaled)

var_3 = pca.explained_variance_ratio_
df_pca = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(K_PCA)])
df_pca["op_status"] = df["op_status"].values

print(f"3-PC model variance explained: {var_3.sum()*100:.1f}%")
print(f"  PC1: {var_3[0]*100:.1f}% | PC2: {var_3[1]*100:.1f}% | PC3: {var_3[2]*100:.1f}%")
print()

# ── Loadings table ────────────────────────────────────────────────────────────
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(K_PCA)],
    index=FEATURES
).round(3)
print("── Loadings Matrix (PC1–PC3) ─────────────────────────────")
display(loadings)

print("\n── Physical interpretation ─────────────────────────────")
print("  PC1 = MECHANICAL DEGRADATION AXIS (55.3%)")
print("    High: vibration_rms (+0.395), vibration_hf (+0.400), bearing_temp (+0.388)")
print("    Low : lube_pressure (−0.363), lube_flow (−0.328), power_factor (−0.333)")
print("    → PC1 directly measures machinery health: high = degraded")
print("  PC2 = ELECTRICAL SUPPLY AXIS (10.0%)")
print("    Dominated by: voltage_v (+0.973)")
print("  PC3 = SPEED AXIS (9.6%)")
print("    Dominated by: speed_rpm (+0.918)")

In [ ]:
# ── Loadings heatmap ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    loadings, annot=True, fmt=".3f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor="#e0e0e0",
    cbar_kws={"label": "Loading value"}, ax=ax
)
ax.set_title(
    "PCA Loadings — Contribution of each sensor to PC1–PC3",
    fontsize=12, fontweight="bold", pad=14
)
ax.set_xlabel("Principal Component")
ax.set_ylabel("Sensor")
plt.tight_layout()
plt.show()

# Biplot summary bar chart
fig2, ax2 = plt.subplots(figsize=(9, 4))
x_pos = np.arange(len(FEATURES))
width = 0.28
colors = [C_BLUE, C_AMBER, C_GREEN]
for i, (col, color) in enumerate(zip(["PC1", "PC2", "PC3"], colors)):
    ax2.bar(x_pos + i * width, loadings[col], width=width, color=color,
            label=col, alpha=0.85, edgecolor="white")
ax2.axhline(0, color="gray", lw=0.8)
ax2.set_xticks(x_pos + width)
ax2.set_xticklabels([f.replace("_", "\n") for f in FEATURES], fontsize=8, rotation=0)
ax2.set_ylabel("Loading Value", fontsize=10)
ax2.set_title("Sensor Contribution to Each Principal Component",
              fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── PC1–PC2 Health Map ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: colored by op_status
for status, color in STATUS_COLORS.items():
    mask = df_pca["op_status"] == status
    ax1.scatter(df_pca.loc[mask, "PC1"], df_pca.loc[mask, "PC2"],
                c=color, label=status, alpha=0.45, s=15, edgecolors="none")
ax1.axhline(0, color="gray", lw=0.8, ls="--", alpha=0.5)
ax1.axvline(0, color="gray", lw=0.8, ls="--", alpha=0.5)
ax1.set_xlabel(f"PC1 — Mechanical Degradation Axis ({var_3[0]*100:.1f}%)", fontsize=10)
ax1.set_ylabel(f"PC2 — Electrical Supply Axis ({var_3[1]*100:.1f}%)", fontsize=10)
ax1.set_title("Motor Health Map — PC1 vs PC2\n(colored by known operating status)",
              fontsize=12, fontweight="bold")
ax1.legend(title="Status", fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: PC1 vs PC3
for status, color in STATUS_COLORS.items():
    mask = df_pca["op_status"] == status
    ax2.scatter(df_pca.loc[mask, "PC1"], df_pca.loc[mask, "PC3"],
                c=color, label=status, alpha=0.45, s=15, edgecolors="none")
ax2.axhline(0, color="gray", lw=0.8, ls="--", alpha=0.5)
ax2.axvline(0, color="gray", lw=0.8, ls="--", alpha=0.5)
ax2.set_xlabel(f"PC1 — Mechanical Degradation ({var_3[0]*100:.1f}%)", fontsize=10)
ax2.set_ylabel(f"PC3 — Speed Axis ({var_3[2]*100:.1f}%)", fontsize=10)
ax2.set_title("Motor Health Map — PC1 vs PC3",
              fontsize=12, fontweight="bold")
ax2.legend(title="Status", fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle(
    "PCA Health Space — Critical motors drift along PC1 (mechanical degradation axis)",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

print("  → Critical motors (red) cluster at high PC1 values — consistent with")
print("    the PC1 interpretation: high PC1 = high vibration + high temp + low lube.")
print("  → PC2 (voltage) and PC3 (speed) add orthogonal context.")

## 🧠 Section 9 — Cluster Profiling & Reliability Index (IR)

PCA enables two complementary health metrics:

**1. Squared Prediction Error (SPE)**  
Measures how poorly the 3-PC model can *reconstruct* the original reading.  
A high SPE means the observation contains patterns not captured by the normal-mode PCs —  
i.e., it's operating in an unexpected regime.

**2. Hotelling T² statistic**  
Measures the Mahalanobis distance from the origin in PC space.  
A high T² means the motor is far from where it normally operates in the health space.

Both are normalized and combined into a single **Reliability Index (IR)**:

$$IR = \frac{1}{1 + \text{SPE}_{norm} + T^2_{norm}} \in (0, 1]$$

**IR interpretation (calibrated to dataset distribution):**

| IR Range | Status | Action |
|---|---|---|
| ≥ 0.86 | 🟢 Normal | Continue operation |
| 0.74 – 0.86 | 🟡 Alert | Monitor closely — inspect at next opportunity |
| 0.62 – 0.74 | 🟠 Severe | Schedule inspection within 24–48h |
| < 0.62 | 🔴 Critical | Halt for immediate maintenance |

In [ ]:
# ── Compute SPE, Hotelling T2, and Reliability Index ─────────────────────────

# SPE: squared reconstruction error in original (scaled) space
X_reconstructed = pca.inverse_transform(X_pca)
SPE = np.sum((X_scaled - X_reconstructed) ** 2, axis=1)

# Hotelling T2: Mahalanobis distance squared in PC space
var_components = np.var(X_pca, axis=0)
T2 = np.sum((X_pca ** 2) / var_components, axis=1)

# Normalize to [0, 1]
SPE_norm = (SPE - SPE.min()) / (SPE.max() - SPE.min())
T2_norm  = (T2  - T2.min())  / (T2.max()  - T2.min())

# Reliability Index
IR = 1 / (1 + SPE_norm + T2_norm)

df["SPE"] = SPE.round(4)
df["T2"]  = T2.round(4)
df["IR"]  = IR.round(4)

print("── IR by operating status ───────────────────────────────")
for status in ["Normal", "Alert", "Critical"]:
    sub = df[df["op_status"] == status]["IR"]
    print(f"  {status:<10}: mean={sub.mean():.4f} | std={sub.std():.4f} | "
          f"p25={sub.quantile(0.25):.4f} | p75={sub.quantile(0.75):.4f}")

print(f"\n── Control thresholds ─────────────────────────────────")
print(f"  SPE 95th percentile : {np.percentile(SPE, 95):.4f}")
print(f"  T2  95th percentile : {np.percentile(T2, 95):.4f}")
print(f"  IR  calibration     : ≥0.86 Normal | ≥0.74 Alert | ≥0.62 Severe | <0.62 Critical")

In [ ]:
# ── SPE and T2 control charts ─────────────────────────────────────────────────
idx = np.arange(len(df))
SPE_UCL = np.percentile(SPE, 95)
T2_UCL  = np.percentile(T2, 95)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7))

# SPE chart
colors_plot = [STATUS_COLORS[s] for s in df["op_status"]]
ax1.scatter(idx, SPE, c=colors_plot, alpha=0.4, s=6, edgecolors="none")
ax1.axhline(SPE_UCL, color=C_RED, lw=1.8, ls="--",
            label=f"UCL 95% = {SPE_UCL:.2f}")
ax1.set_ylabel("SPE (Reconstruction Error)", fontsize=10)
ax1.set_title("Squared Prediction Error (SPE) Control Chart", fontsize=11, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.25)

# T2 chart
ax2.scatter(idx, T2, c=colors_plot, alpha=0.4, s=6, edgecolors="none")
ax2.axhline(T2_UCL, color=C_RED, lw=1.8, ls="--",
            label=f"UCL 95% = {T2_UCL:.2f}")
ax2.set_xlabel("Observation Index", fontsize=10)
ax2.set_ylabel("Hotelling T² Statistic", fontsize=10)
ax2.set_title("Hotelling T² Control Chart", fontsize=11, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.25)

# Legend patches
patches = [mpatches.Patch(color=c, label=s) for s, c in STATUS_COLORS.items()]
fig.legend(handles=patches, loc="lower right", fontsize=9,
           title="Operating Status", bbox_to_anchor=(0.98, 0.01))

plt.suptitle("Multivariate Statistical Process Control — PCA-Based Health Monitoring",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

above_SPE = (SPE > SPE_UCL).sum()
above_T2  = (T2 > T2_UCL).sum()
print(f"  Points above SPE UCL: {above_SPE} ({above_SPE/len(df)*100:.1f}%)")
print(f"  Points above T2 UCL : {above_T2} ({above_T2/len(df)*100:.1f}%)")

In [ ]:
# ── IR distribution by status ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: IR density by status
for status, color in STATUS_COLORS.items():
    sub = df[df["op_status"] == status]["IR"]
    ax1.hist(sub, bins=40, alpha=0.65, color=color, label=status,
             density=True, edgecolor="none")
for thresh, color in [(0.86, C_GREEN), (0.74, C_AMBER), (0.62, C_ORANGE)]:
    ax1.axvline(x=thresh, color=color, ls="--", lw=1.8, alpha=0.9)
ax1.set_xlabel("Reliability Index (IR)", fontsize=11)
ax1.set_ylabel("Density", fontsize=11)
ax1.set_title("IR Distribution by Operating Status",
              fontsize=12, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(axis="y", alpha=0.3)

# Right: IR on PC1 axis
sc = ax2.scatter(df_pca["PC1"], df_pca["PC2"], c=IR, cmap="RdYlGn",
                 vmin=0.5, vmax=1.0, alpha=0.6, s=14, edgecolors="none")
plt.colorbar(sc, ax=ax2, label="Reliability Index (IR)")
ax2.set_xlabel(f"PC1 — Mechanical Degradation ({var_3[0]*100:.1f}%)", fontsize=10)
ax2.set_ylabel(f"PC2 — Electrical Supply ({var_3[1]*100:.1f}%)", fontsize=10)
ax2.set_title("IR Heat Map on PC1–PC2 Health Space\n(Green = healthy · Red = degraded)",
              fontsize=12, fontweight="bold")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# IR summary
print("── IR classification summary ────────────────────────────")
ir_labels = pd.cut(IR, bins=[0, 0.62, 0.74, 0.86, 1.0],
                   labels=["Critical", "Severe", "Alert", "Normal"])
print(ir_labels.value_counts().to_string())

## ✅ Section 10 — Clustering Validation & Stability

PCA validation in a predictive maintenance context uses three layers:

| Validation | Method | What it tells us |
|---|---|---|
| **Variance coverage** | Cumulative explained variance | Are 3 PCs sufficient? |
| **Discriminative power** | IR separation across status groups | Does IR track degradation? |
| **Control threshold alignment** | SPE/T2 vs status distribution | Are UCL limits physically meaningful? |
| **Reconstruction fidelity** | Feature correlation of X vs X̂ | How much signal is preserved? |

In [ ]:
# ── Variance coverage ─────────────────────────────────────────────────────────
print("── 1. Variance Coverage ────────────────────────────────")
print(f"  3-PC model covers: {var_3.sum()*100:.1f}% of total sensor variance")
print(f"  Residual (SPE space): {(1-var_3.sum())*100:.1f}% — captured in reconstruction error")

# ── IR discriminative power ────────────────────────────────────────────────────
print("\n── 2. IR Discriminative Power ──────────────────────────")
ir_normal   = df[df["op_status"]=="Normal"]["IR"].mean()
ir_alert    = df[df["op_status"]=="Alert"]["IR"].mean()
ir_critical = df[df["op_status"]=="Critical"]["IR"].mean()
print(f"  Normal mean IR   : {ir_normal:.4f}")
print(f"  Alert mean IR    : {ir_alert:.4f}")
print(f"  Critical mean IR : {ir_critical:.4f}")
print(f"  Gradient: Normal→Critical separation = {(ir_normal-ir_critical):.4f} IR units")

# ── PC1 separation ────────────────────────────────────────────────────────────
print("\n── 3. PC1 Separation by Status ─────────────────────────")
for status in ["Normal", "Alert", "Critical"]:
    sub = df_pca[df_pca["op_status"]==status]["PC1"]
    print(f"  {status:<10}: PC1 mean={sub.mean():.2f} | std={sub.std():.2f}")

# ── Reconstruction fidelity ────────────────────────────────────────────────────
print("\n── 4. Reconstruction Fidelity (3 PCs) ──────────────────")
X_recon = pca.inverse_transform(X_pca)
for i, feat in enumerate(FEATURES):
    corr = np.corrcoef(X_scaled[:, i], X_recon[:, i])[0, 1]
    flag = "✅" if corr > 0.85 else ("⚠️ " if corr > 0.70 else "❌")
    print(f"  {flag} {feat:<30}: r={corr:.4f}")

# ── Control threshold alignment ────────────────────────────────────────────────
print("\n── 5. UCL Coverage by Status ─────────────────────────")
for status in ["Normal", "Alert", "Critical"]:
    sub = df[df["op_status"]==status]
    pct_spe = (sub["SPE"] > SPE_UCL).mean() * 100
    pct_t2  = (sub["T2"]  > T2_UCL).mean() * 100
    print(f"  {status:<10}: {pct_spe:.1f}% above SPE UCL | {pct_t2:.1f}% above T2 UCL")

In [ ]:
# ── Validation summary table ──────────────────────────────────────────────────
summary = pd.DataFrame({
    "Metric": [
        "Variance (3 PCs)", "PC1 Variance",
        "IR Normal mean", "IR Alert mean", "IR Critical mean",
        "PC1 separation (Normal→Critical)",
        "SPE UCL (95th pct)", "T2 UCL (95th pct)",
    ],
    "Value": [
        f"{var_3.sum()*100:.1f}%", f"{var_3[0]*100:.1f}%",
        f"{ir_normal:.4f}", f"{ir_alert:.4f}", f"{ir_critical:.4f}",
        f"{ir_normal-ir_critical:.4f} IR units",
        f"{SPE_UCL:.4f}", f"{T2_UCL:.4f}",
    ],
    "Interpretation": [
        "Sufficient for health monitoring",
        "Single dominant health axis",
        "Well within normal range",
        "Intermediate — correctly identified",
        "Below threshold — correctly flagged",
        "Clear IR gradient across degradation levels",
        "Calibrated from normal operating data",
        "Calibrated from normal operating data",
    ],
})
display(summary)

## 🧪 Section 11 — Motor Health Classifier & Operational Playbook

The fitted PCA model (scaler + PCA object) classifies any new motor snapshot in real time.  
Given 10 sensor readings, the classifier returns:
- **PC1, PC2, PC3** — position in health space
- **SPE** — reconstruction error (unexplained pattern)
- **T²** — distance from normal operating center
- **IR** — combined Reliability Index with traffic-light status
- **Recommended action** — calibrated to the IR band

**Three representative scenarios:**

| Scenario | Conditions | Expected IR |
|---|---|---|
| **A — Baseline** | All sensors within nominal operating range | ≥ 0.86 🟢 |
| **B — Alert** | Moderate vibration elevation, slight temp rise | 0.74–0.86 🟡 |
| **C — Critical** | High vibration, elevated temp, degraded lubrication | < 0.62 🔴 |

In [ ]:
# ── Motor health classifier function ─────────────────────────────────────────
def classify_motor(sensor_readings: dict, verbose: bool = True) -> dict:
    """
    Classify a new motor condition snapshot using the PCA health model.

    Parameters
    ----------
    sensor_readings : dict with keys matching FEATURES
    verbose         : print classification report

    Returns
    -------
    dict with PC scores, SPE, T2, IR, status, and recommended action
    """
    x = np.array([[sensor_readings[f] for f in FEATURES]])
    x_sc  = scaler.transform(x)
    x_pca = pca.transform(x_sc)
    x_rec = pca.inverse_transform(x_pca)

    spe    = float(np.sum((x_sc - x_rec) ** 2))
    t2     = float(np.sum((x_pca ** 2) / var_components))
    spe_n  = (spe - SPE.min()) / (SPE.max() - SPE.min())
    t2_n   = (t2  - T2.min())  / (T2.max()  - T2.min())
    ir     = 1 / (1 + spe_n + t2_n)

    if   ir >= IR_THRESHOLDS["Normal"]: status = "Normal";   icon = "🟢"; action = "Continue operation. Log reading."
    elif ir >= IR_THRESHOLDS["Alert"]:  status = "Alert";    icon = "🟡"; action = "Monitor closely. Inspect at next planned stop."
    elif ir >= IR_THRESHOLDS["Severe"]: status = "Severe";   icon = "🟠"; action = "Schedule inspection within 48 hours."
    else:                               status = "Critical";  icon = "🔴"; action = "Halt for immediate maintenance."

    result = {
        "PC1": round(x_pca[0, 0], 3),
        "PC2": round(x_pca[0, 1], 3),
        "PC3": round(x_pca[0, 2], 3),
        "SPE": round(spe, 4),
        "T2":  round(t2, 4),
        "IR":  round(ir, 4),
        "status": status,
    }

    if verbose:
        print(f"{'═' * 55}")
        print(f"  {icon}  MOTOR STATUS: {status.upper()}")
        print(f"{'═' * 55}")
        print(f"  Reliability Index (IR) : {ir:.4f}")
        print(f"  PC1 (Mechanical)       : {x_pca[0,0]:+.3f}")
        print(f"  PC2 (Electrical)       : {x_pca[0,1]:+.3f}")
        print(f"  PC3 (Speed)            : {x_pca[0,2]:+.3f}")
        print(f"  SPE (Reconstruction)   : {spe:.4f} (UCL={SPE_UCL:.4f})")
        print(f"  T²  (Distance)         : {t2:.4f}  (UCL={T2_UCL:.4f})")
        print(f"  Recommended action     : {action}")
        print()
    return result

print("✅ Classifier loaded — ready for scenarios.")
print(f"   IR thresholds: Normal≥{IR_THRESHOLDS['Normal']} · Alert≥{IR_THRESHOLDS['Alert']} · "
      f"Severe≥{IR_THRESHOLDS['Severe']} · Critical<{IR_THRESHOLDS['Severe']}")

In [ ]:
# ── Scenario A: Baseline Normal ───────────────────────────────────────────────
print("Scenario A — Baseline Normal Operation")
r_A = classify_motor(dict(zip(FEATURES, [
    2.00, 0.50, 65.0, 18.0, 460.0, 95.0, 1480.0, 35.0, 3.50, 25.0
])))

In [ ]:
# ── Scenario B: Alert Condition ────────────────────────────────────────────────
print("Scenario B — Alert: Moderate vibration + temp elevation")
r_B = classify_motor(dict(zip(FEATURES, [
    4.13, 1.35, 84.59, 18.97, 454.06, 93.92, 1430.7, 29.1, 3.18, 22.19
])))

In [ ]:
# ── Scenario C: Critical — Severe Degradation ────────────────────────────────
print("Scenario C — Critical: High vibration, elevated temp, degraded lubrication")
r_C = classify_motor(dict(zip(FEATURES, [
    6.78, 2.00, 90.09, 28.75, 456.81, 84.62, 1484.82, 28.78, 2.30, 42.34
])))

# ── Summary plot ──────────────────────────────────────────────────────────────
scenarios = {"A Normal": r_A, "B Alert": r_B, "C Critical": r_C}
sc_colors = {"A Normal": C_GREEN, "B Alert": C_AMBER, "C Critical": C_RED}

fig, ax = plt.subplots(figsize=(8, 3.5))
ir_vals  = [r["IR"] for r in scenarios.values()]
sc_names = list(scenarios.keys())
colors   = list(sc_colors.values())
bars = ax.barh(sc_names, ir_vals, color=colors, alpha=0.85, edgecolor="white", height=0.45)
for bar, val in zip(bars, ir_vals):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"IR = {val:.4f}", va="center", fontsize=10, fontweight="bold")
for thresh, color, label in [(0.86, C_GREEN, "0.86"), (0.74, C_AMBER, "0.74"), (0.62, C_ORANGE, "0.62")]:
    ax.axvline(x=thresh, color=color, ls="--", lw=1.5, alpha=0.9)
    ax.text(thresh - 0.005, -0.55, label, color=color, fontsize=8, ha="right")
ax.set_xlim(0.4, 1.08)
ax.set_xlabel("Reliability Index (IR)", fontsize=11)
ax.set_title("Health Classifier — Scenario Comparison",
             fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

---

## 💡 Final Reflection

PCA's power in predictive maintenance is not dimensionality reduction per se —  
it's the ability to define a **normal operating manifold** and then measure  
how far any new observation has drifted from it.

Key takeaways from Project 17:

1. **PC1 = 54.4% of variance = the motor health axis.**  
   One number — the PC1 score — summarizes the collective signal of vibration,  
   temperature, current, and lubrication in a single coordinate.

2. **SPE and T² measure different types of deviation.**  
   T² flags motors that are in an unusual *known* state.  
   SPE flags motors exhibiting a pattern the PCA model has never seen —  
   which can be an early warning of a new failure mode.

3. **The IR is a KPI, not just a number.**  
   Track it over time. A motor with IR trending from 0.90 → 0.85 → 0.78 → 0.70  
   is telling you a story before the alarm fires.

4. **PCA does not require failure data.**  
   The entire model is built from normal operation history.  
   That makes it deployable in new equipment from day one.

5. **Loadings are the root cause map.**  
   When IR drops, look at which sensor contributed most to PC1.  
   The loading matrix tells you which sensor is pulling the score away from normal.

---

**What to try next:**
- Add a **time-series trend** of IR — rolling 7-day average as an early warning indicator
- Combine PCA with **K-Means** (Project 15) to cluster motors by degradation stage
- Deploy the classifier via **OPC-UA** to your SCADA system for real-time monitoring

---

*LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa*